# 02 ? Feature engineering and chronological split

Build leakage-safe features, persist the full feature table, and create chronological train/validation/test partitions. Requires notebook 01 to have completed successfully.

In [ ]:
## Setup

import os

import sys, warnings, yaml
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
_candidates.extend(path.parent for path in _cwd.glob("*/configs/pipeline.yaml"))
PROJECT_ROOT = next((path for path in _candidates if (path / "configs" / "pipeline.yaml").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find configs/pipeline.yaml. Open the project folder or set the notebook working directory to this repository.")
os.chdir(PROJECT_ROOT)
CONFIG_PATH = PROJECT_ROOT / "configs" / "pipeline.yaml"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.spark_session   import get_spark
from src.features.pipeline    import build_feature_pipeline
from src.models.splitter      import chronological_split
from src.visualisation.eda_plots import plot_correlation_heatmap

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    CFG = yaml.safe_load(f)

PROCESSED  = PROJECT_ROOT / CFG['artifacts']['processed_dir']
FIGURES    = PROJECT_ROOT / CFG['reports']['figures_dir']
for d in [PROCESSED, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)
with open(PROCESSED / 'audit_meta.yaml') as f:
    META = yaml.safe_load(f)

TS_COL     = META['ts_col']
TARGET_COL = f'target_log_return_{CFG["features"]["target_horizon"]}'
PASSTHROUGH = {'open','high','low','close','volume', TS_COL}

print(f'Timestamp column : {TS_COL}')
print(f'Target column    : {TARGET_COL}')

---
## 1 · Feature Engineering

> ⚠️ **Anti-leakage**: all window functions use `rowsBetween(-(w-1), 0)` (past-only). Target label built with `F.lead()`.

In [ ]:
# ── Load cleaned data → Spark ─────────────────────────────────────────
df_clean = pd.read_parquet(PROCESSED / 'btc_cleaned.parquet')
df_clean[TS_COL] = pd.to_datetime(df_clean[TS_COL])

spark = get_spark()
sdf   = spark.createDataFrame(df_clean)
print(f'Loaded into Spark: {sdf.count():,} rows × {len(sdf.columns)} cols')

In [ ]:
# ── Run full feature pipeline ─────────────────────────────────────────
print('Building features (PySpark window functions)...')
sdf_features = build_feature_pipeline(sdf, ts_col=TS_COL, close_col='close')

print(f'Feature DataFrame: {len(sdf_features.columns)} columns')

In [ ]:
# ── Persist features without collecting the full dataset ───────────────
feature_cols = [c for c in sdf_features.columns
                if c not in PASSTHROUGH and not c.startswith('target_')]
feature_path = PROCESSED / 'btc_features.parquet'
sdf_features.write.mode('overwrite').parquet(str(feature_path))
sdf_features = spark.read.parquet(str(feature_path))

print(f'\nFeature groups:')
for label, kw in [('Returns', 'return'), ('Volatility', 'vol'),
                  ('Volume', 'volume'), ('Market Structure', 'drawdown momentum slope range')]:
    cols = [c for c in feature_cols if any(k in c for k in kw.split())]
    print(f'  {label:<18}: {len(cols)} cols')
print(f'  Total features  : {len(feature_cols)}')

In [ ]:
# ── Feature × target correlation heatmap (top 20) ────────────────────
top_cols = feature_cols[:20] + [TARGET_COL]
corr_sample = (sdf_features.select(*top_cols)
               .sample(withReplacement=False, fraction=0.05, seed=42)
               .limit(50000).toPandas())
fig = plot_correlation_heatmap(corr_sample, columns=top_cols,
                               title='Feature × Target Correlation (top 20 features)')
fig.write_html(str(FIGURES / 'feature_correlation.html'), include_plotlyjs='cdn')


In [ ]:
print(f'Saved Spark feature dataset → {feature_path}  '
      f'({len(sdf_features.columns)} columns)')

---
## 2 · Chronological Train / Val / Test Split

**Rule:** Strictly chronological — no random shuffling, ever.

In [ ]:
# ── Split ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F

train_df, val_df, test_df = chronological_split(sdf_features, ts_col=TS_COL)

print('\nSplit summary:')
for name, part in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    bounds = part.agg(F.min(TS_COL).alias('start'), F.max(TS_COL).alias('end')).first()
    print(f'  {name:5}: {bounds.start.date()} → {bounds.end.date()}  '
          f'({part.count():,} rows)')

In [ ]:
# ── Prepare Spark partitions (drop NaN rows) ──────────────────────────
all_model_cols = feature_cols + [TARGET_COL, TS_COL]
train_clean = train_df.select(*all_model_cols).dropna(subset=all_model_cols)
val_clean   = val_df.select(*all_model_cols).dropna(subset=all_model_cols)
test_clean  = test_df.select(*all_model_cols).dropna(subset=all_model_cols)

for name, part in [('Train', train_clean), ('Val', val_clean), ('Test', test_clean)]:
    print(f'{name}: {part.count():,} rows × {len(all_model_cols)} columns')

In [ ]:
# ── Target distribution per partition ────────────────────────────────
fig = go.Figure()
for seed, (name, part, col) in enumerate([
        ('Train', train_clean, '#4FC3F7'),
        ('Val', val_clean, '#F7931A'),
        ('Test', test_clean, '#66BB6A')]):
    y_sample = (part.select(TARGET_COL)
                .sample(withReplacement=False, fraction=0.1, seed=seed)
                .limit(50000).toPandas()[TARGET_COL])
    fig.add_trace(go.Histogram(x=y_sample, name=f'{name} sample', histnorm='probability density',
                               opacity=0.6, marker_color=col))
fig.update_layout(barmode='overlay', title=f'Target Distribution — {TARGET_COL}',
                  template='plotly_dark', xaxis_title='Log Return')
fig.write_html(str(FIGURES / 'target_distribution.html'), include_plotlyjs='cdn')


In [ ]:
# ── Save partitions + metadata ────────────────────────────────────────
train_clean.write.mode('overwrite').parquet(str(PROCESSED / 'train.parquet'))
val_clean.write.mode('overwrite').parquet(str(PROCESSED / 'val.parquet'))
test_clean.write.mode('overwrite').parquet(str(PROCESSED / 'test.parquet'))

def max_date(part):
    return part.agg(F.max(TS_COL).alias('end')).first()['end'].date()

split_meta = {
    'feature_cols' : feature_cols,
    'target_col'   : TARGET_COL,
    'ts_col'       : TS_COL,
    'train_end'    : str(max_date(train_clean)),
    'val_end'      : str(max_date(val_clean)),
    'test_end'     : str(max_date(test_clean)),
}
with open(PROCESSED / 'split_meta.yaml', 'w') as f:
    yaml.dump(split_meta, f, default_flow_style=False)

print('Partitions saved.')
print('Next → 03_modelling.ipynb')
spark.stop()
print('Spark stopped.')